## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading/loading the **FEI Morph v2** dataset.

In [1]:
import cv2
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
print("Starting the download of the FEI Morph dataset...")
print("Note: This might take a while the first time...")

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
fei_morph_path = BASE_DIR /"FEI Morph v2"
print("Dataset path:", fei_morph_path) if fei_morph_path.is_dir() else print("Dataset not founded")

fei_face_path = BASE_DIR /"FEI Face"
print("Dataset path:", fei_face_path if fei_face_path.is_dir() else print("Dataset not founded"))

Starting the download of the FEI Morph dataset...
Note: This might take a while the first time...
Dataset path: /Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2
Dataset path: /Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face


In [3]:
folders = os.listdir(fei_morph_path)
tot_elements = len(list(fei_morph_path.iterdir()))
print(f"Total elements inside { fei_morph_path}: {tot_elements}")
print("Contents inside", fei_morph_path, ": ")
print('\n'.join([f"- {f}" for f in folders[:10]]))

Total elements inside /Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2: 14000
Contents inside /Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2 : 
- M_82-11_49-11_C03_B30_W30_PA03_PM00_F00.png
- M_109-11_9-11_C03_B50_W50_PA03_PM00_F00.png
- M_121-11_128-11_C02_B50_W50_PA02_PM00_F00.png
- M_121-11_170-11_C15_B50_W50_PA15_PM00_F00.png
- M_194-11_112-11_C15_B30_W30_PA15_PM00_F00.png
- M_62-11_54-11_C02_B30_W30_PA02_PM00_F00.png
- M_122-11_184-11_C15_B30_W30_PA15_PM00_F00.png
- M_155-11_162-11_C05_B30_W30_PA05_PM00_F00.png
- M_20-11_13-11_C05_B50_W50_PA05_PM00_F00.png
- M_109-11_53-11_C01_B50_W50_PA01_PM00_F00.png


In [4]:
folders = os.listdir(fei_face_path)
print("Contents inside", fei_face_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

Contents inside /Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face : 
- originalimages_part2
- originalimages_part3
- originalimages_part4
- .DS_Store
- originalimages_part1


In [5]:
subfolder_name = folders[-1]
subfolder_path = os.path.join(fei_face_path, subfolder_name)
if os.path.isdir(subfolder_path):
    contents = os.listdir(subfolder_path)
    print(f"Contents inside the first subfolder: {subfolder_name}")
    print('\n'.join([f"- {f}" for f in contents[:10]]))

Contents inside the first subfolder: originalimages_part1
- 25-01.jpg
- 22-08.jpg
- 8-07.jpg
- 8-13.jpg
- 6-02.jpg
- 35-04.jpg
- 35-10.jpg
- 14-09.jpg
- 48-04.jpg
- 13-14.jpg


## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all images from the dataset.

Columns:
- `filename`: filename
- `path`: file path 
- `label`: 0 = real/original, 1 = fake/morphed
- `subj1`: ID of the first subject 
- `subj2`: ID of the second subject; NaN if the image is an original
- `pose_number`: Pose identifier for Subject 1 
- `pose_number_2`: Pose identifier for Subject 2 
- `algorithm`: morphing algorithm used or "original"
- `morph_factor_1`: blending percentage of Subject 1 
- `morph_factor_2`: blending percentage of Subject 2 
- `post_proc_auto`: automated post-processing 
- `post_proc_manual`: manual retouching or professional editing.
- `digital_ps`: digital alterations or Photoshop-style enhancements.
- `resolution`: image dimensions

In [6]:
def get_resolution(file_path):
    with Image.open(file_path) as img:
        width, height = img.size
        resolution = f"{width}x{height}"
    return resolution

morph_data = []

for file_path in fei_morph_path.glob("*.png"):
    parts = file_path.stem.split("_")

    if len(parts) == 9:
        
        resolution = get_resolution(file_path)

        subj1_raw = parts[1]
        subj2_raw = parts[2]
        
        subj1 = subj1_raw.split("-")[0] if "-" in subj1_raw else subj1_raw
        pose_num = subj1_raw.split("-")[1] if "-" in subj1_raw else None
        
        subj2 = subj2_raw.split("-")[0] if "-" in subj2_raw else subj2_raw
        pose_num_2 = subj2_raw.split("-")[1] if "-" in subj2_raw else None
                
        morph_data.append({
            "filename": file_path.name,
            "path": str(file_path),
            "label": 1,
            "subj1": int(subj1),
            "subj2": int(subj2),
            "pose_number": pose_num,
            "pose_number_2": pose_num_2,
            "algorithm": parts[3],
            "morph_factor_1": parts[4],
            "morph_factor_2": parts[5],
            "post_proc_auto": parts[6],
            "post_proc_manual": parts[7],
            "digital_ps": parts[8],
            "resolution": resolution
        })
    
real_data = []

for file_path in fei_face_path.rglob("*.jpg"):

    parts = file_path.stem.split("-")
    
    if len(parts) == 2:
        subj_id = parts[0]
        pose_num = parts[1]
    

        resolution = get_resolution(file_path)

        real_data.append({
            "filename": file_path.name,
            "path": str(file_path),
            "label": 0,
            "subj1": int(subj_id),
            "subj2": pd.NA,
            "pose_number": pose_num,
            "pose_number_2": None,
            "algorithm": "original",
            "morph_factor_1": None,
            "morph_factor_2": None,
            "post_proc_auto": None,
            "post_proc_manual": None,
            "digital_ps": None,
            "resolution": resolution
        })

morphed = pd.DataFrame(morph_data)
originals = pd.DataFrame(real_data)

pd.set_option('display.max_colwidth', None)
display(morphed.sample(20))
display(originals.head())

,filename,path,label,subj1,subj2,pose_number,pose_number_2,algorithm,morph_factor_1,morph_factor_2,post_proc_auto,post_proc_manual,digital_ps,resolution
9475,M_18-11_186-11_C05_B50_W50_PA05_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_18-11_186-11_C05_B50_W50_PA05_PM00_F00.png,1,18,186,11,11,C05,B50,W50,PA05,PM00,F00,380x507
2269,M_99-11_198-11_C05_B30_W30_PA05_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_99-11_198-11_C05_B30_W30_PA05_PM00_F00.png,1,99,198,11,11,C05,B30,W30,PA05,PM00,F00,376x501
6226,M_119-11_179-11_C01_B30_W30_PA01_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_119-11_179-11_C01_B30_W30_PA01_PM00_F00.png,1,119,179,11,11,C01,B30,W30,PA01,PM00,F00,500x600
10233,M_193-11_54-11_C15_B50_W50_PA15_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_193-11_54-11_C15_B50_W50_PA15_PM00_F00.png,1,193,54,11,11,C15,B50,W50,PA15,PM00,F00,640x480
1721,M_140-11_183-11_C05_B50_W50_PA05_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_140-11_183-11_C05_B50_W50_PA05_PM00_F00.png,1,140,183,11,11,C05,B50,W50,PA05,PM00,F00,404x539
13822,M_181-11_168-11_C16_B30_W30_PA16_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_181-11_168-11_C16_B30_W30_PA16_PM00_F00.png,1,181,168,11,11,C16,B30,W30,PA16,PM00,F00,1511x1943
5618,M_151-11_143-11_C15_B50_W50_PA15_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_151-11_143-11_C15_B50_W50_PA15_PM00_F00.png,1,151,143,11,11,C15,B50,W50,PA15,PM00,F00,640x480
7589,M_5-11_44-11_C08_B30_W30_PA08_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_5-11_44-11_C08_B30_W30_PA08_PM00_F00.png,1,5,44,11,11,C08,B30,W30,PA08,PM00,F00,1511x1943
7397,M_139-11_156-11_C05_B50_W50_PA05_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_139-11_156-11_C05_B50_W50_PA05_PM00_F00.png,1,139,156,11,11,C05,B50,W50,PA05,PM00,F00,340x453
2971,M_88-11_80-11_C01_B50_W50_PA01_PM00_F00.png,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Morph v2/M_88-11_80-11_C01_B50_W50_PA01_PM00_F00.png,1,88,80,11,11,C01,B50,W50,PA01,PM00,F00,500x600


,filename,path,label,subj1,subj2,pose_number,pose_number_2,algorithm,morph_factor_1,morph_factor_2,post_proc_auto,post_proc_manual,digital_ps,resolution
0,77-09.jpg,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face/originalimages_part2/77-09.jpg,0,77,<NA>,09,None,original,None,None,None,None,None,640x480
1,70-14.jpg,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face/originalimages_part2/70-14.jpg,0,70,<NA>,14,None,original,None,None,None,None,None,640x480
2,81-06.jpg,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face/originalimages_part2/81-06.jpg,0,81,<NA>,06,None,original,None,None,None,None,None,640x480
3,81-12.jpg,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face/originalimages_part2/81-12.jpg,0,81,<NA>,12,None,original,None,None,None,None,None,640x480
4,58-01.jpg,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI Face/originalimages_part2/58-01.jpg,0,58,<NA>,01,None,original,None,None,None,None,None,640x480


In [7]:
print("--- POSE BALANCING ---")

print(f"Originals before filtering: {len(originals)}") #(Should be 2800)
originals = originals[originals['pose_number'] == '11']

print(f"Originals after filtering: {len(originals)}") #(Should be 200, one per subject)

full_fei_df = pd.concat([morphed, originals], ignore_index=True)

print(f"\nDataset MERGED and BALANCED successfully!")
print(f"Total images: {len(full_fei_df)} (Morphed: {len(morphed)} + Originals: {len(originals)})") #(Should be 14200, one per subject)

--- POSE BALANCING ---
Originals before filtering: 2800
Originals after filtering: 200

Dataset MERGED and BALANCED successfully!
Total images: 14200 (Morphed: 14000 + Originals: 200)


### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly:

* **Total number of images**
* **Label distribution (Fake vs Original)**: analysis of the class balance
* **Morphing Method Distribution**: breakdown of images by the algorithm used
* **Identity Consistency Check**: verification that `subj1` and `subj2` are always different in morphing rows to ensure no "self-morphs" (which would technically be originals) exist in the fake category
* **Unique Identity Count**: identification of all unique individuals (200 in total) involved in the dataset
* **Missing Values Check**: validation of the DataFrame's integrity. We expect `subj2` to have null values for original images, but other critical columns (like `path` or `label`) must be fully populated
* **Post-processing Automation (PA) Levels**: verification of the distribution of automated enhancements applied to the images, which can range from no processing (PA00) to various levels of digital adjustment

In [8]:
print("--- SANITY CHECK ON FULL DATASET ---")

print(f"Total images: {len(full_fei_df)}")
print(f"\nLabel distribution (Fake vs Original):\n{full_fei_df['label'].value_counts()}")

print(f"\nMorphing Method (Algorithm) distribution:\n{full_fei_df['algorithm'].value_counts(dropna=False)}")

fakes_only = full_fei_df[full_fei_df['label'] == 'fake']
identities_match = (fakes_only['subj1'] == fakes_only['subj2']).any()
print(f"\nAre there any cases where Subject 1 == Subject 2 in morphs? {'YES' if identities_match else 'NO'}")

all_identities = set(full_fei_df['subj1'].dropna()).union(set(full_fei_df['subj2'].dropna()))
print(f"\nTotal unique individuals in the dataset: {len(all_identities)}")
print(f"Unique identities (Source 1): {full_fei_df['subj1'].nunique()}")
print(f"Unique identities (Source 2): {full_fei_df['subj2'].nunique()}")

print(f"\nChecking for missing values:\n{full_fei_df.isnull().sum()}")

print(f"\nPost-processing automation levels (PA):\n{full_fei_df['post_proc_auto'].value_counts(dropna=False)}")

--- SANITY CHECK ON FULL DATASET ---
Total images: 14200

Label distribution (Fake vs Original):
label
1    14000
0      200
Name: count, dtype: int64

Morphing Method (Algorithm) distribution:
algorithm
C03         2000
C02         2000
C15         2000
C05         2000
C01         2000
C16         2000
C08         2000
original     200
Name: count, dtype: int64

Are there any cases where Subject 1 == Subject 2 in morphs? NO

Total unique individuals in the dataset: 200
Unique identities (Source 1): 200
Unique identities (Source 2): 200

Checking for missing values:
filename              0
path                  0
label                 0
subj1                 0
subj2               200
pose_number           0
pose_number_2       200
algorithm             0
morph_factor_1      200
morph_factor_2      200
post_proc_auto      200
post_proc_manual    200
digital_ps          200
resolution            0
dtype: int64

Post-processing automation levels (PA):
post_proc_auto
PA03    2000
PA02    

### 2.2  - Distribution of Fake Images per Subject (Subj1)

Analyzes how many fake images exist per subj1 to identify if some identities dominate the fake samples

In [9]:
fake_df = full_fei_df[full_fei_df["label"] == 1]
per_target = fake_df.groupby("subj1").size()
print("\n--- FAKE PER SUBJ1 STATISTICS ---")
print(per_target.describe())


--- FAKE PER SUBJ1 STATISTICS ---
count    197.00000
mean      71.06599
std       41.56751
min       14.00000
25%       42.00000
50%       56.00000
75%       98.00000
max      252.00000
dtype: float64


### 2.3 - Distribution Of Methods per Subject (Subj1) (Check Variability):

Creates a pivot table showing how many images of each algorithm exist per subj1 to verify that each identity has a representative set of algorithm and to detect identities with too few or missing algorithm 
types

In [10]:
pivot = pd.pivot_table(
    full_fei_df,
    index="subj1",
    columns="algorithm",
    values="filename",
    aggfunc="count",
    fill_value=0
)

print("\n--- ALGORITHM DISTRIBUTION PER SUB1 ---")
display(pivot.sample(20))


--- ALGORITHM DISTRIBUTION PER SUB1 ---


algorithm,C01,C02,C03,C05,C08,C15,C16,original
subj1,,,,,,,,
155,4,4,4,4,4,4,4,1
118,6,6,6,6,6,6,6,1
70,8,8,8,8,8,8,8,1
181,6,6,6,6,6,6,6,1
93,16,16,16,16,16,16,16,1
162,8,8,8,8,8,8,8,1
49,14,14,14,14,14,14,14,1
151,20,20,20,20,20,20,20,1
154,8,8,8,8,8,8,8,1


### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [11]:
print("Normalize label ditribution:")
print(full_fei_df["label"].value_counts(normalize=True))

Normalize label ditribution:
label
1    0.985915
0    0.014085
Name: proportion, dtype: float64


### Dataset Observations

Based on the sanity check results, several key characteristics of the merged FEI Morph dataset emerge.

* **Extreme Class Imbalance**: The dataset is heavily skewed towards morphed images. There are **14,000 Fakes (98.6%)** compared to only **200 Originals (1.4%)**.
* **Perfect Algorithm & Pipeline Symmetry**: The 14,000 morphs are perfectly distributed across 7 different morphing algorithms (`C01`, `C02`, `C03`, `C05`, `C08`, `C15`, `C16`), with exactly **2,000 images per algorithm**. Additionally, the Post-Processing Automation (PA) levels map 1:1 with these algorithms.
* **Logical Missing Values**: There are exactly 200 missing values in columns like `subj2`, `morph_factor`, and `post_proc_auto`. This confirms perfect data integrity, as these 200 rows correspond exactly to the 200 "Original" images (which naturally do not have a second subject or morphing parameters).
* **Identity Integrity**: There are 200 unique individuals in the dataset, and the check confirms there are **zero instances of self-morphing** (`Subject 1 == Subject 2` is False across the board).
* **Subject Variance in Morphs**: While the overall algorithms are balanced, the frequency of a specific person acting as the base face (`subj1`) varies significantly. Some subjects act as the base for up to **252 morphs**, while others are used only **14 times** (mean $\approx$ 71). 
* **Intra-Subject Algorithm Balance**: Despite the variance in how often a subject is used as `subj1`, the distribution of algorithms *for that specific subject* is always perfectly balanced. For example, if Subject 63 is used for 154 morphs, there are exactly 22 images for each of the 7 algorithms.

## 3 - Face Extraction & Preprocessing

This section processes the raw images to extract the faces, standardize their dimensions, and remove irrelevant background information.
### Processing Pipeline:
* **SSD Face Detection**: We use OpenCV's DNN (ResNet-10 SSD) to detect faces. Detections are cropped with a 15% margin and padded into a square to prevent aspect-ratio distortion.
* **Fallback Method:** If the model fails to confidently detect a face, we automatically apply a static center crop.
* **Standardization**: Every extracted face is resized to **224x224 pixels**, which is the standard input size for most modern vision backbones.
* **Metadata Tracking**: The processed images are saved in a new `FEI_Processed` directory. A new DataFrame (`full_fei_processed_df`) is generated, tracking the new file paths and adding a `detection_type` column (`ssd` or `cc` for center crop) to monitor the extraction quality.

In [12]:
# CONFIGURATION 
processed_root = "FEI_Processed"
target_size = (224, 224)
confidence_threshold = 0.6

prototxt_path = "models/deploy.prototxt"
model_path = "models/res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)

def process_face(image_path, target_size):
    frame = cv2.imread(image_path)
    if frame is None: return None, "error"
    
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()

    processed_frame = None
    method = "cc" 

    # Face Detection via SSD
    if detections.shape[2] > 0:
        i = np.argmax(detections[0, 0, :, 2])
        confidence = detections[0, 0, i, 2]
        
        if confidence > confidence_threshold:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")
            
            margin = int(max(x2-x1, y2-y1) * 0.2)
            cx1, cy1 = max(0, x1-margin), max(0, y1-margin)
            cx2, cy2 = min(w, x2+margin), min(h, y2+margin)
            
            face = frame[cy1:cy2, cx1:cx2]
            if face.size > 0:
                fh, fw = face.shape[:2]
                side = max(fh, fw)
                square = np.zeros((side, side, 3), np.uint8)
                square[(side-fh)//2:(side-fh)//2+fh, (side-fw)//2:(side-fw)//2+fw] = face
                processed_frame = cv2.resize(square, target_size)
                method = "ssd"

    # Fallback: Center Crop if SSD fails
    if processed_frame is None:
        min_dim = min(h, w)
        start_x, start_y = (w - min_dim) // 2, (h - min_dim) // 2
        crop = frame[start_y:start_y+min_dim, start_x:start_x+min_dim]
        processed_frame = cv2.resize(crop, target_size)
    
    return processed_frame, method

# MAIN LOOP
for label in ["original", "fake"]:
    os.makedirs(os.path.join(processed_root, label), exist_ok=True)

processed_data = []

print(f"--- STARTING FACE EXTRACTION & RESIZE TO {target_size} ---")

for idx, row in tqdm(full_fei_df.iterrows(), total=len(full_fei_df)):
    img_path = row['path']
    
    val_label = str(row['label'])
    if val_label in ['0', 'original']:
        label_folder = "original"
    elif val_label in ['1', 'fake']:
        label_folder = "fake"
    else:
        label_folder = val_label
    
    face_img, detect_method = process_face(img_path, target_size)
    
    if face_img is not None:
        base_name, extension = os.path.splitext(row['filename'])
        new_filename = f"{base_name}_{detect_method}{extension}"
        new_path = os.path.join(processed_root, label_folder, new_filename)
        
        cv2.imwrite(new_path, face_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        
        new_row = row.to_dict()
        new_row['path'] = new_path 
        new_row['filename'] = new_filename
        new_row['detection_type'] = detect_method
        new_row['resolution'] = f"{target_size[0]}x{target_size[1]}"
        processed_data.append(new_row)

full_fei_processed_df = pd.DataFrame(processed_data)

print(f"\n--- EXTRACTION COMPLETE ---")
print(f"Total processed images: {len(full_fei_processed_df)}")
print(f"Processed images saved in: {processed_root}")
full_fei_processed_df.head()

--- STARTING FACE EXTRACTION & RESIZE TO (224, 224) ---


100%|██████████| 14200/14200 [05:31<00:00, 42.81it/s]


--- EXTRACTION COMPLETE ---
Total processed images: 14200
Processed images saved in: FEI_Processed


,filename,path,label,subj1,subj2,pose_number,pose_number_2,algorithm,morph_factor_1,morph_factor_2,post_proc_auto,post_proc_manual,digital_ps,resolution,detection_type
0,M_82-11_49-11_C03_B30_W30_PA03_PM00_F00_ssd.png,FEI_Processed/fake/M_82-11_49-11_C03_B30_W30_PA03_PM00_F00_ssd.png,1,82,49.0,11,11,C03,B30,W30,PA03,PM00,F00,224x224,ssd
1,M_109-11_9-11_C03_B50_W50_PA03_PM00_F00_ssd.png,FEI_Processed/fake/M_109-11_9-11_C03_B50_W50_PA03_PM00_F00_ssd.png,1,109,9.0,11,11,C03,B50,W50,PA03,PM00,F00,224x224,ssd
2,M_121-11_128-11_C02_B50_W50_PA02_PM00_F00_ssd.png,FEI_Processed/fake/M_121-11_128-11_C02_B50_W50_PA02_PM00_F00_ssd.png,1,121,128.0,11,11,C02,B50,W50,PA02,PM00,F00,224x224,ssd
3,M_121-11_170-11_C15_B50_W50_PA15_PM00_F00_ssd.png,FEI_Processed/fake/M_121-11_170-11_C15_B50_W50_PA15_PM00_F00_ssd.png,1,121,170.0,11,11,C15,B50,W50,PA15,PM00,F00,224x224,ssd
4,M_194-11_112-11_C15_B30_W30_PA15_PM00_F00_ssd.png,FEI_Processed/fake/M_194-11_112-11_C15_B30_W30_PA15_PM00_F00_ssd.png,1,194,112.0,11,11,C15,B30,W30,PA15,PM00,F00,224x224,ssd


## 4 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_images/dataframe_images`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [13]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "fei_images.csv")
full_fei_processed_df.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")

--- SAVING DATAFRAME ---
Data succesfully saved to: ./processed_images/fei_images.csv
